# State Cluster Transitions and Structural Stability
Research question :

2. How did structural similarities among states change during the 2020–2022 shock period and the 2023–2024 post-shock period? 
3. Which states remained structurally stable, temporarily changed, or experienced a possible structural shift across the three periods? 


## Step 5G — Baseline, Shock, and Post-Shock Comparison

This notebook tracks each state's economic-structure cluster across three
periods:

- Baseline: 2015–2019
- Shock: 2020–2022
- Post-shock: 2023–2024

The analysis classifies states as structurally stable, temporarily changed,
possibly structurally shifted, or following a complex transition.

These categories describe changes in relative cluster membership. They do
not prove that the shock caused a permanent structural transformation.

In [ ]:
# ============================================================
# STEP 5G.2 — IMPORT LIBRARIES
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_theme(
    style="whitegrid",
    context="notebook"
)

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_colwidth",
    None
)

pd.set_option(
    "display.float_format",
    "{:,.4f}".format
)

print("Libraries imported successfully.")
print("Pandas version:", pd.__version__)

In [ ]:
# ============================================================
# STEP 5G.3 — DEFINE PROJECT PATHS
# ============================================================

PROJECT_DIR = Path.cwd()

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "results"
)

TABLES_DIR = (
    RESULTS_DIR
    / "tables"
)

FIGURES_DIR = (
    RESULTS_DIR
    / "figures"
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

data_paths = {
    "cluster_assignments":
        PROCESSED_DIR
        / "state_period_cluster_assignments_labeled.csv",

    "cluster_features":
        PROCESSED_DIR
        / "state_period_clusters_with_features_labeled.csv",

    "growth_period":
        PROCESSED_DIR
        / "state_dynamic_growth_period_averages.csv"
}

for dataset_name, file_path in data_paths.items():
    print(
        f"{dataset_name}:",
        "Found" if file_path.exists() else "Missing",
        "—",
        file_path
    )

In [ ]:
# ============================================================
# STEP 5G.4 — LOAD STEP 5G DATASETS
# ============================================================

missing_files = [
    str(file_path)
    for file_path in data_paths.values()
    if not file_path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following required files are missing:\n"
        + "\n".join(missing_files)
    )

cluster_assignments = pd.read_csv(
    data_paths["cluster_assignments"],
    dtype={
        "state_fips": "string"
    }
)

cluster_features = pd.read_csv(
    data_paths["cluster_features"],
    dtype={
        "state_fips": "string"
    }
)

growth_period = pd.read_csv(
    data_paths["growth_period"],
    dtype={
        "state_fips": "string"
    }
)

for dataframe in [
    cluster_assignments,
    cluster_features,
    growth_period
]:
    dataframe["state_fips"] = (
        dataframe["state_fips"]
        .str.zfill(2)
    )

print("Step 5G datasets loaded successfully.")

In [ ]:
# ============================================================
# STEP 5G.5 — DEFINE CONSTANTS
# ============================================================

BASELINE_PERIOD = "Baseline_2015_2019"
SHOCK_PERIOD = "Shock_2020_2022"
POST_SHOCK_PERIOD = "Post_Shock_2023_2024"

PERIOD_ORDER = [
    BASELINE_PERIOD,
    SHOCK_PERIOD,
    POST_SHOCK_PERIOD
]

IDENTIFIER_COLUMNS = [
    "state_fips",
    "state",
    "period"
]

PCA_COMPONENT_NAMES = [
    "PC1",
    "PC2",
    "PC3",
    "PC4"
]

GROWTH_FEATURES = [
    "real_gdp_per_capita_growth",
    "real_income_per_capita_growth",
    "total_employment_growth",
    "real_average_wage_growth"
]

STATUS_ORDER = [
    "Structurally stable",
    "Temporary shock-period change",
    "Possible shock-origin persistent shift",
    "Possible post-shock shift",
    "Complex or continuing transition"
]

for dataframe in [
    cluster_assignments,
    cluster_features,
    growth_period
]:
    dataframe["period"] = pd.Categorical(
        dataframe["period"],
        categories=PERIOD_ORDER,
        ordered=True
    )

cluster_assignments = (
    cluster_assignments
    .sort_values(
        ["state_fips", "period"]
    )
    .reset_index(drop=True)
)

cluster_features = (
    cluster_features
    .sort_values(
        ["state_fips", "period"]
    )
    .reset_index(drop=True)
)

growth_period = (
    growth_period
    .sort_values(
        ["state_fips", "period"]
    )
    .reset_index(drop=True)
)

print("Period and feature constants defined.")

In [ ]:
# ============================================================
# STEP 5G.6 — VALIDATE LOADED DATA
# ============================================================

required_assignment_columns = (
    IDENTIFIER_COLUMNS
    + PCA_COMPONENT_NAMES
    + [
        "Cluster_ID",
        "Cluster",
        "Cluster_Name"
    ]
)

missing_assignment_columns = [
    column
    for column in required_assignment_columns
    if column not in cluster_assignments.columns
]

if missing_assignment_columns:
    raise KeyError(
        "Missing cluster-assignment columns: "
        f"{missing_assignment_columns}"
    )

missing_growth_columns = [
    column
    for column in GROWTH_FEATURES
    if column not in growth_period.columns
]

if missing_growth_columns:
    raise KeyError(
        "Missing growth columns: "
        f"{missing_growth_columns}"
    )

# Validate cluster assignments
assert cluster_assignments.shape[0] == 150
assert cluster_assignments["state_fips"].nunique() == 50
assert cluster_assignments["period"].nunique() == 3
assert cluster_assignments["Cluster_ID"].nunique() == 7

assert not cluster_assignments.duplicated(
    ["state_fips", "period"]
).any()

assert cluster_assignments[
    required_assignment_columns
].notna().all().all()

# Validate supporting datasets
assert cluster_features.shape[0] == 150
assert growth_period.shape[0] == 150

assert not cluster_features.duplicated(
    ["state_fips", "period"]
).any()

assert not growth_period.duplicated(
    ["state_fips", "period"]
).any()

# Confirm that all datasets contain identical state-period keys
assignment_keys = set(
    cluster_assignments[
        ["state_fips", "period"]
    ]
    .astype(str)
    .itertuples(
        index=False,
        name=None
    )
)

feature_keys = set(
    cluster_features[
        ["state_fips", "period"]
    ]
    .astype(str)
    .itertuples(
        index=False,
        name=None
    )
)

growth_keys = set(
    growth_period[
        ["state_fips", "period"]
    ]
    .astype(str)
    .itertuples(
        index=False,
        name=None
    )
)

assert assignment_keys == feature_keys
assert assignment_keys == growth_keys

print("All Step 5G dataset validations passed.")

In [ ]:
# ============================================================
# STEP 5G.7 — CREATE CLUSTER-NAME LOOKUP
# ============================================================

cluster_name_lookup = (
    cluster_assignments[
        [
            "Cluster_ID",
            "Cluster",
            "Cluster_Name"
        ]
    ]
    .drop_duplicates()
    .sort_values("Cluster_ID")
    .reset_index(drop=True)
)

assert cluster_name_lookup.shape[0] == 7

assert (
    cluster_name_lookup
    .groupby("Cluster_ID")[
        "Cluster_Name"
    ]
    .nunique()
    .eq(1)
    .all()
)

CLUSTER_NAME_MAP = dict(
    zip(
        cluster_name_lookup["Cluster_ID"],
        cluster_name_lookup["Cluster_Name"]
    )
)

display(cluster_name_lookup)

In [ ]:
# ============================================================
# STEP 5G.8 — CREATE STATE CLUSTER PATHWAYS
# ============================================================

state_cluster_pathways = (
    cluster_assignments
    .pivot(
        index=[
            "state_fips",
            "state"
        ],
        columns="period",
        values="Cluster_ID"
    )
    .reindex(
        columns=PERIOD_ORDER
    )
    .reset_index()
)

state_cluster_pathways = (
    state_cluster_pathways
    .rename(
        columns={
            BASELINE_PERIOD:
                "Baseline_Cluster_ID",

            SHOCK_PERIOD:
                "Shock_Cluster_ID",

            POST_SHOCK_PERIOD:
                "Post_Shock_Cluster_ID"
        }
    )
)

cluster_id_columns = [
    "Baseline_Cluster_ID",
    "Shock_Cluster_ID",
    "Post_Shock_Cluster_ID"
]

assert state_cluster_pathways.shape[0] == 50

assert state_cluster_pathways[
    cluster_id_columns
].notna().all().all()

for column in cluster_id_columns:
    state_cluster_pathways[column] = (
        state_cluster_pathways[column]
        .astype(int)
    )

state_cluster_pathways[
    "Baseline_Cluster_Name"
] = state_cluster_pathways[
    "Baseline_Cluster_ID"
].map(CLUSTER_NAME_MAP)

state_cluster_pathways[
    "Shock_Cluster_Name"
] = state_cluster_pathways[
    "Shock_Cluster_ID"
].map(CLUSTER_NAME_MAP)

state_cluster_pathways[
    "Post_Shock_Cluster_Name"
] = state_cluster_pathways[
    "Post_Shock_Cluster_ID"
].map(CLUSTER_NAME_MAP)

state_cluster_pathways[
    "Cluster_Path"
] = (
    state_cluster_pathways[
        "Baseline_Cluster_ID"
    ].astype(str)
    + " → "
    + state_cluster_pathways[
        "Shock_Cluster_ID"
    ].astype(str)
    + " → "
    + state_cluster_pathways[
        "Post_Shock_Cluster_ID"
    ].astype(str)
)

display(
    state_cluster_pathways.head()
)

In [ ]:
# ============================================================
# STEP 5G.9 — CLASSIFY STATE STRUCTURAL PATHWAYS
# ============================================================

baseline_cluster = (
    state_cluster_pathways[
        "Baseline_Cluster_ID"
    ]
)

shock_cluster = (
    state_cluster_pathways[
        "Shock_Cluster_ID"
    ]
)

post_shock_cluster = (
    state_cluster_pathways[
        "Post_Shock_Cluster_ID"
    ]
)

classification_conditions = [
    # A → A → A
    (
        baseline_cluster.eq(shock_cluster)
        & shock_cluster.eq(post_shock_cluster)
    ),

    # A → B → A
    (
        baseline_cluster.ne(shock_cluster)
        & post_shock_cluster.eq(baseline_cluster)
    ),

    # A → B → B
    (
        baseline_cluster.ne(shock_cluster)
        & post_shock_cluster.eq(shock_cluster)
    ),

    # A → A → B
    (
        baseline_cluster.eq(shock_cluster)
        & post_shock_cluster.ne(baseline_cluster)
    )
]

classification_labels = [
    "Structurally stable",
    "Temporary shock-period change",
    "Possible shock-origin persistent shift",
    "Possible post-shock shift"
]

state_cluster_pathways[
    "Structural_Status"
] = np.select(
    classification_conditions,
    classification_labels,
    default="Complex or continuing transition"
)

state_cluster_pathways[
    "Structural_Status"
] = pd.Categorical(
    state_cluster_pathways[
        "Structural_Status"
    ],
    categories=STATUS_ORDER,
    ordered=True
)

state_cluster_pathways.columns.name = None

display(
    state_cluster_pathways[
        [
            "state",
            "Cluster_Path",
            "Structural_Status"
        ]
    ]
    .sort_values(
        [
            "Structural_Status",
            "state"
        ]
    )
)

In [ ]:
# ============================================================
# STEP 5G.10 — CLASSIFICATION DEFINITIONS
# ============================================================

structural_status_definitions = pd.DataFrame({
    "Example_Path": [
        "A → A → A",
        "A → B → A",
        "A → B → B",
        "A → A → B",
        "A → B → C"
    ],
    "Structural_Status": STATUS_ORDER,
    "Interpretation": [
        (
            "The state remained in the same structural "
            "cluster across all three periods."
        ),
        (
            "The state changed during the shock period "
            "and returned to its baseline cluster."
        ),
        (
            "The state changed during the shock period "
            "and remained in that cluster afterward."
        ),
        (
            "The state remained stable during the shock "
            "but changed during the post-shock period."
        ),
        (
            "The state followed three different or otherwise "
            "more complicated cluster positions."
        )
    ]
})

display(structural_status_definitions)

In [ ]:
# ============================================================
# STEP 5G.11 — SUMMARIZE STRUCTURAL STATUS
# ============================================================

structural_status_summary = (
    state_cluster_pathways[
        "Structural_Status"
    ]
    .value_counts(sort=False)
    .rename_axis("Structural_Status")
    .reset_index(name="Number_of_States")
)

structural_status_summary[
    "Percent_of_States"
] = (
    structural_status_summary[
        "Number_of_States"
    ]
    / 50
    * 100
)

display(
    structural_status_summary.round(2)
)

In [ ]:
# ============================================================
# STEP 5G.12 — LIST STATES BY STRUCTURAL STATUS
# ============================================================

states_by_structural_status = (
    state_cluster_pathways
    .groupby(
        "Structural_Status",
        observed=False
    )
    .agg(
        Number_of_States=(
            "state",
            "size"
        ),
        States=(
            "state",
            lambda states: ", ".join(
                sorted(states)
            )
        )
    )
    .reset_index()
)

display(states_by_structural_status)

In [ ]:
# ============================================================
# STEP 5G.13 — PERIOD-TO-PERIOD CHANGE SUMMARY
# ============================================================

period_change_records = [
    {
        "Comparison": "Baseline → Shock",
        "Changed_States": int(
            baseline_cluster.ne(
                shock_cluster
            ).sum()
        )
    },
    {
        "Comparison": "Shock → Post-shock",
        "Changed_States": int(
            shock_cluster.ne(
                post_shock_cluster
            ).sum()
        )
    },
    {
        "Comparison": "Baseline → Post-shock",
        "Changed_States": int(
            baseline_cluster.ne(
                post_shock_cluster
            ).sum()
        )
    }
]

period_change_summary = pd.DataFrame(
    period_change_records
)

period_change_summary[
    "Unchanged_States"
] = (
    50
    - period_change_summary[
        "Changed_States"
    ]
)

period_change_summary[
    "Changed_Percent"
] = (
    period_change_summary[
        "Changed_States"
    ]
    / 50
    * 100
)

display(
    period_change_summary.round(2)
)

In [ ]:
# ============================================================
# STEP 5G.14 — CREATE TRANSITION MATRICES
# ============================================================

baseline_to_shock_matrix = pd.crosstab(
    state_cluster_pathways[
        "Baseline_Cluster_ID"
    ],
    state_cluster_pathways[
        "Shock_Cluster_ID"
    ]
).reindex(
    index=range(7),
    columns=range(7),
    fill_value=0
)

shock_to_post_matrix = pd.crosstab(
    state_cluster_pathways[
        "Shock_Cluster_ID"
    ],
    state_cluster_pathways[
        "Post_Shock_Cluster_ID"
    ]
).reindex(
    index=range(7),
    columns=range(7),
    fill_value=0
)

baseline_to_post_matrix = pd.crosstab(
    state_cluster_pathways[
        "Baseline_Cluster_ID"
    ],
    state_cluster_pathways[
        "Post_Shock_Cluster_ID"
    ]
).reindex(
    index=range(7),
    columns=range(7),
    fill_value=0
)

print("Baseline → Shock")
display(baseline_to_shock_matrix)

print("Shock → Post-shock")
display(shock_to_post_matrix)

print("Baseline → Post-shock")
display(baseline_to_post_matrix)

Values on the diagonal represent states remaining in the same cluster. Off-diagonal values represent cluster changes.

In [ ]:
# ============================================================
# STEP 5G.15 — VISUALIZE CLUSTER TRANSITIONS
# ============================================================

transition_matrices = [
    (
        baseline_to_shock_matrix,
        "Baseline → Shock"
    ),
    (
        shock_to_post_matrix,
        "Shock → Post-shock"
    ),
    (
        baseline_to_post_matrix,
        "Baseline → Post-shock"
    )
]

fig, axes = plt.subplots(
    1,
    3,
    figsize=(19, 5.5)
)

for axis, (
    transition_matrix,
    transition_title
) in zip(
    axes,
    transition_matrices
):
    sns.heatmap(
        transition_matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        linewidths=0.5,
        square=True,
        ax=axis
    )

    axis.set_title(
        transition_title,
        fontsize=13
    )

    axis.set_xlabel(
        "Later-period cluster"
    )

    axis.set_ylabel(
        "Earlier-period cluster"
    )

plt.suptitle(
    "State Economic-Structure Cluster Transitions",
    fontsize=16,
    y=1.03
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 5G.16 — VISUALIZE STRUCTURAL STATUS
# ============================================================

STATUS_COLORS = {
    "Structurally stable": "#4E79A7",
    "Temporary shock-period change": "#F28E2B",
    "Possible shock-origin persistent shift": "#B07AA1",
    "Possible post-shock shift": "#76B7B2",
    "Complex or continuing transition": "#79706E"
}

status_plot_data = (
    structural_status_summary
    .sort_values(
        "Number_of_States",
        ascending=True
    )
)

plt.figure(figsize=(11, 6))

bars = plt.barh(
    status_plot_data[
        "Structural_Status"
    ].astype(str),
    status_plot_data[
        "Number_of_States"
    ],
    color=[
        STATUS_COLORS[status]
        for status in status_plot_data[
            "Structural_Status"
        ].astype(str)
    ]
)

plt.bar_label(
    bars,
    labels=[
        (
            f"{number} "
            f"({percent:.1f}%)"
        )
        for number, percent in zip(
            status_plot_data[
                "Number_of_States"
            ],
            status_plot_data[
                "Percent_of_States"
            ]
        )
    ],
    padding=5
)

plt.title(
    "State Structural Stability Across Three Periods",
    fontsize=14
)

plt.xlabel("Number of states")
plt.ylabel("")

plt.xlim(
    0,
    status_plot_data[
        "Number_of_States"
    ].max() * 1.25
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 5G.17 — JOIN STRUCTURAL STATUS TO GROWTH DATA
# ============================================================

# This prepares supporting economic-growth interpretation without using growth variables to redefine the clusters.

transition_growth_data = (
    growth_period
    .merge(
        state_cluster_pathways[
            [
                "state_fips",
                "state",
                "Cluster_Path",
                "Structural_Status"
            ]
        ],
        on=[
            "state_fips",
            "state"
        ],
        how="left",
        validate="many_to_one"
    )
)

assert transition_growth_data.shape[0] == 150

assert transition_growth_data[
    "Structural_Status"
].notna().all()

growth_by_status_and_period = (
    transition_growth_data
    .groupby(
        [
            "Structural_Status",
            "period"
        ],
        observed=True
    )[GROWTH_FEATURES]
    .mean()
)

display(
    growth_by_status_and_period.round(3)
)

In [ ]:
# ============================================================
# STEP 5G.18 — SAVE TRANSITION OUTPUTS
# ============================================================

state_cluster_pathways.to_csv(
    PROCESSED_DIR
    / "state_cluster_transition_pathways.csv",
    index=False
)

structural_status_summary.to_csv(
    TABLES_DIR
    / "structural_status_summary.csv",
    index=False
)

states_by_structural_status.to_csv(
    TABLES_DIR
    / "states_by_structural_status.csv",
    index=False
)

period_change_summary.to_csv(
    TABLES_DIR
    / "period_to_period_cluster_changes.csv",
    index=False
)

baseline_to_shock_matrix.to_csv(
    TABLES_DIR
    / "transition_matrix_baseline_to_shock.csv"
)

shock_to_post_matrix.to_csv(
    TABLES_DIR
    / "transition_matrix_shock_to_post.csv"
)

baseline_to_post_matrix.to_csv(
    TABLES_DIR
    / "transition_matrix_baseline_to_post.csv"
)

growth_by_status_and_period.to_csv(
    TABLES_DIR
    / "growth_by_structural_status_and_period.csv"
)

print("Step 5G outputs saved successfully.")

In [ ]:
display(structural_status_summary.round(2))
display(states_by_structural_status)
display(period_change_summary.round(2))
display(state_cluster_pathways)

## Step 5G conclusions

State economic and industry-employment structures were predominantly
stable across the baseline, shock, and post-shock periods. Forty states,
representing 80% of the sample, remained in the same structural cluster
throughout all three periods.

Nine states—California, Colorado, Kansas, Michigan, New Hampshire,
North Carolina, Oregon, Tennessee, and Washington—changed clusters
between the 2015–2019 baseline and the 2020–2022 shock period. All nine
remained in their new clusters during 2023–2024. These pathways are
classified **as possible shock-origin persistent shifts**.

**Maine** was the only state that remained in its baseline cluster during
2020–2022 but changed clusters during 2023–2024. It is classified **as a
possible post-shock shift**.

No state exhibited a temporary shock-period change by moving to a
different cluster and then returning to its baseline cluster. No state
followed a complex three-cluster transition.

The transition analysis indicates that interstate structural similarities
were largely stable. Most changes occurred between the baseline and shock
periods, and those changes generally persisted during the available
post-shock period.

These results are descriptive. A persistent cluster change means that a
state became relatively closer to a different multivariate cluster
centroid. **It does not prove that the economic shock caused a permanent
structural transformation.** The 2023–2024 post-shock window is also too
short to establish long-term permanence.

# Step 5H — Illinois Economic Peer Analysis

Illinois's closest economic peers are identified separately during the
baseline, shock, and post-shock periods.

Similarity is measured using Euclidean distance across PC1–PC4. These four
components preserve 91.82% of the standardized structural information.

A smaller distance indicates that another state has a structural profile
more similar to Illinois. Cluster membership is shown as supporting
context but does not determine the ranking.

In [ ]:
# ============================================================
# STEP 5H.1 — DEFINE PEER-ANALYSIS SETTINGS
# ============================================================

TARGET_STATE = "Illinois"
TOP_N_PEERS = 5

PCA_COMPONENT_NAMES = [
    "PC1",
    "PC2",
    "PC3",
    "PC4"
]

PERIOD_DISPLAY_LABELS = {
    BASELINE_PERIOD: "Baseline: 2015–2019",
    SHOCK_PERIOD: "Shock: 2020–2022",
    POST_SHOCK_PERIOD: "Post-shock: 2023–2024"
}

print("Target state:", TARGET_STATE)
print("Peers retained per period:", TOP_N_PEERS)
print("Distance dimensions:", PCA_COMPONENT_NAMES)

In [ ]:
# ============================================================
# STEP 5H.2 — VALIDATE ILLINOIS OBSERVATIONS
# ============================================================

illinois_observations = (
    cluster_assignments.loc[
        cluster_assignments["state"].eq(
            TARGET_STATE
        ),
        IDENTIFIER_COLUMNS
        + PCA_COMPONENT_NAMES
        + [
            "Cluster_ID",
            "Cluster",
            "Cluster_Name"
        ]
    ]
    .sort_values("period")
    .reset_index(drop=True)
)

assert illinois_observations.shape[0] == 3

assert (
    illinois_observations["period"]
    .nunique()
    == 3
)

assert illinois_observations[
    PCA_COMPONENT_NAMES
].notna().all().all()

display(
    illinois_observations
)

For state `j`, the distance from Illinois is:

$$
d_{IL,j}
=
\sqrt{
(PC1_{IL}-PC1_j)^2
+
(PC2_{IL}-PC2_j)^2
+
(PC3_{IL}-PC3_j)^2
+
(PC4_{IL}-PC4_j)^2
}
$$

In [ ]:
# ============================================================
# STEP 5H.3 — CALCULATE ILLINOIS-TO-STATE DISTANCES
# ============================================================

illinois_peer_records = []

for period in PERIOD_ORDER:

    period_data = (
        cluster_assignments.loc[
            cluster_assignments[
                "period"
            ].astype(str).eq(period)
        ]
        .copy()
        .reset_index(drop=True)
    )

    assert period_data.shape[0] == 50

    illinois_row = period_data.loc[
        period_data["state"].eq(
            TARGET_STATE
        )
    ]

    assert illinois_row.shape[0] == 1

    illinois_coordinates = (
        illinois_row[
            PCA_COMPONENT_NAMES
        ]
        .iloc[0]
        .to_numpy(dtype=float)
    )

    illinois_cluster_id = int(
        illinois_row[
            "Cluster_ID"
        ].iloc[0]
    )

    illinois_cluster_name = (
        illinois_row[
            "Cluster_Name"
        ].iloc[0]
    )

    peer_coordinates = (
        period_data[
            PCA_COMPONENT_NAMES
        ]
        .to_numpy(dtype=float)
    )

    period_data["PCA_Distance"] = np.sqrt(
        np.sum(
            (
                peer_coordinates
                - illinois_coordinates
            ) ** 2,
            axis=1
        )
    )

    # Exclude Illinois from its own peer ranking
    period_peers = (
        period_data.loc[
            ~period_data["state"].eq(
                TARGET_STATE
            )
        ]
        .copy()
        .sort_values(
            [
                "PCA_Distance",
                "state"
            ]
        )
        .reset_index(drop=True)
    )

    period_peers["Peer_Rank"] = (
        np.arange(
            1,
            len(period_peers) + 1
        )
    )

    period_peers["Same_Cluster_as_Illinois"] = (
        period_peers["Cluster_ID"]
        .eq(illinois_cluster_id)
    )

    period_peers["Illinois_Cluster_ID"] = (
        illinois_cluster_id
    )

    period_peers["Illinois_Cluster_Name"] = (
        illinois_cluster_name
    )

    illinois_peer_records.append(
        period_peers[
            [
                "state_fips",
                "state",
                "period",
                "Peer_Rank",
                "PCA_Distance",
                "Same_Cluster_as_Illinois",
                "Cluster_ID",
                "Cluster_Name",
                "Illinois_Cluster_ID",
                "Illinois_Cluster_Name"
            ]
            + PCA_COMPONENT_NAMES
        ]
    )

illinois_peer_distances = pd.concat(
    illinois_peer_records,
    ignore_index=True
)

assert illinois_peer_distances.shape[0] == (
    49 * 3
)

print(
    "Peer-distance observations:",
    len(illinois_peer_distances)
)

In [ ]:
# ============================================================
# STEP 5H.4 — EXTRACT TOP ILLINOIS PEERS
# ============================================================

illinois_top_peers = (
    illinois_peer_distances.loc[
        illinois_peer_distances[
            "Peer_Rank"
        ].le(TOP_N_PEERS)
    ]
    .copy()
)

illinois_top_peers["Period_Label"] = (
    illinois_top_peers["period"]
    .astype(str)
    .map(PERIOD_DISPLAY_LABELS)
)

display(
    illinois_top_peers[
        [
            "Period_Label",
            "Peer_Rank",
            "state",
            "PCA_Distance",
            "Same_Cluster_as_Illinois",
            "Cluster_ID",
            "Cluster_Name"
        ]
    ]
    .sort_values(
        [
            "Peer_Rank"
        ]
    )
    .round(4)
)

In [ ]:
# ============================================================
# STEP 5H.5 — SUMMARIZE PEERS BY PERIOD
# ============================================================

illinois_peer_summary = (
    illinois_top_peers
    .sort_values(
        [
            "period",
            "Peer_Rank"
        ]
    )
    .groupby(
        "period",
        observed=True
    )
    .agg(
        Closest_Peer=(
            "state",
            "first"
        ),
        Closest_Distance=(
            "PCA_Distance",
            "first"
        ),
        Top_Five_Peers=(
            "state",
            lambda states: ", ".join(states)
        ),
        Same_Cluster_Peers=(
            "Same_Cluster_as_Illinois",
            "sum"
        )
    )
    .reindex(PERIOD_ORDER)
    .reset_index()
)

illinois_peer_summary["Period_Label"] = (
    illinois_peer_summary["period"]
    .astype(str)
    .map(PERIOD_DISPLAY_LABELS)
)

display(
    illinois_peer_summary[
        [
            "Period_Label",
            "Closest_Peer",
            "Closest_Distance",
            "Top_Five_Peers",
            "Same_Cluster_Peers"
        ]
    ].round(4)
)

In [ ]:
# ============================================================
# STEP 5H.6 — IDENTIFY RECURRING PEERS
# ============================================================

recurring_illinois_peers = (
    illinois_top_peers
    .groupby(
        [
            "state_fips",
            "state"
        ]
    )
    .agg(
        Periods_in_Top_Five=(
            "period",
            "nunique"
        ),
        Best_Rank=(
            "Peer_Rank",
            "min"
        ),
        Mean_Rank=(
            "Peer_Rank",
            "mean"
        ),
        Mean_PCA_Distance=(
            "PCA_Distance",
            "mean"
        ),
        Periods=(
            "Period_Label",
            lambda values: ", ".join(
                values
            )
        )
    )
    .reset_index()
    .sort_values(
        [
            "Periods_in_Top_Five",
            "Mean_Rank",
            "Mean_PCA_Distance"
        ],
        ascending=[
            False,
            True,
            True
        ]
    )
)

display(
    recurring_illinois_peers.round(4)
)

In [ ]:
# ============================================================
# STEP 5H.7 — COMPARE PEER GROUPS ACROSS PERIODS
# ============================================================

top_peer_sets = {
    period: set(
        illinois_top_peers.loc[
            illinois_top_peers[
                "period"
            ].astype(str).eq(period),
            "state"
        ]
    )
    for period in PERIOD_ORDER
}

peer_period_comparisons = [
    (
        BASELINE_PERIOD,
        SHOCK_PERIOD,
        "Baseline → Shock"
    ),
    (
        SHOCK_PERIOD,
        POST_SHOCK_PERIOD,
        "Shock → Post-shock"
    ),
    (
        BASELINE_PERIOD,
        POST_SHOCK_PERIOD,
        "Baseline → Post-shock"
    )
]

peer_overlap_records = []

for (
    earlier_period,
    later_period,
    comparison_name
) in peer_period_comparisons:

    earlier_peers = top_peer_sets[
        earlier_period
    ]

    later_peers = top_peer_sets[
        later_period
    ]

    shared_peers = (
        earlier_peers
        & later_peers
    )

    combined_peers = (
        earlier_peers
        | later_peers
    )

    peer_overlap_records.append({
        "Comparison": comparison_name,
        "Shared_Top_Five_Peers":
            len(shared_peers),
        "Shared_States":
            ", ".join(
                sorted(shared_peers)
            ),
        "Jaccard_Similarity": (
            len(shared_peers)
            / len(combined_peers)
        )
    })

illinois_peer_overlap = pd.DataFrame(
    peer_overlap_records
)

display(
    illinois_peer_overlap.round(4)
)

### Jaccard Similarity

The Jaccard similarity is:

$$
J(A,B)
=
\frac{|A \cap B|}{|A \cup B|}
$$

where:

- $A$ = peer list from one period
- $B$ = peer list from another period
- $A \cap B$ = peers shared by both lists
- $A \cup B$ = all unique peers across both lists

### Interpretation

- **$J(A,B) = 1.0$**: identical peer lists
- **$J(A,B) = 0.0$**: no shared peers
- **Higher Jaccard similarity**: greater stability in the peer structure
- **Lower Jaccard similarity**: greater change in the peer structure

In [ ]:
# ============================================================
# STEP 5H.8 — VISUALIZE ILLINOIS'S TOP FIVE PEERS
# ============================================================

from matplotlib.patches import Patch

PEER_COLORS = {
    True: "#4E79A7",
    False: "#F28E2B"
}

maximum_peer_distance = (
    illinois_top_peers[
        "PCA_Distance"
    ].max()
    * 1.18
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 6),
    sharex=True
)

for axis, period in zip(
    axes,
    PERIOD_ORDER
):

    period_plot_data = (
        illinois_top_peers.loc[
            illinois_top_peers[
                "period"
            ].astype(str).eq(period)
        ]
        .sort_values(
            "PCA_Distance",
            ascending=False
        )
    )

    bars = axis.barh(
        period_plot_data["state"],
        period_plot_data["PCA_Distance"],
        color=[
            PEER_COLORS[
                same_cluster
            ]
            for same_cluster in period_plot_data[
                "Same_Cluster_as_Illinois"
            ]
        ]
    )

    axis.bar_label(
        bars,
        labels=[
            f"{distance:.2f}"
            for distance in period_plot_data[
                "PCA_Distance"
            ]
        ],
        padding=3,
        fontsize=9
    )

    axis.set_title(
        PERIOD_DISPLAY_LABELS[period]
    )

    axis.set_xlabel(
        "Four-component PCA distance"
    )

    axis.set_xlim(
        0,
        maximum_peer_distance
    )

    axis.grid(
        axis="x",
        alpha=0.3
    )

axes[0].set_ylabel("Peer state")
axes[1].set_ylabel("")
axes[2].set_ylabel("")

legend_elements = [
    Patch(
        facecolor=PEER_COLORS[True],
        label="Same cluster as Illinois"
    ),
    Patch(
        facecolor=PEER_COLORS[False],
        label="Different cluster"
    )
]

fig.legend(
    handles=legend_elements,
    loc="lower center",
    ncol=2,
    bbox_to_anchor=(0.5, -0.02)
)

plt.suptitle(
    "Illinois's Closest Economic Peers by Period",
    fontsize=16,
    y=1.02
)

plt.tight_layout(
    rect=[0, 0.07, 1, 1]
)

plt.show()

In [ ]:
# ============================================================
# STEP 5H.9 — VISUALIZE PEER-RANK CONSISTENCY
# ============================================================

peer_rank_matrix = (
    illinois_top_peers
    .pivot(
        index="state",
        columns="period",
        values="Peer_Rank"
    )
    .reindex(
        columns=PERIOD_ORDER
    )
)

peer_rank_order = (
    recurring_illinois_peers[
        "state"
    ]
    .tolist()
)

peer_rank_matrix = (
    peer_rank_matrix
    .reindex(peer_rank_order)
)

peer_rank_matrix.columns = [
    PERIOD_DISPLAY_LABELS[str(period)]
    for period in peer_rank_matrix.columns
]

plt.figure(
    figsize=(
        10,
        max(
            5,
            len(peer_rank_matrix) * 0.45
        )
    )
)

sns.heatmap(
    peer_rank_matrix,
    annot=True,
    fmt=".0f",
    cmap="YlOrBr_r",
    vmin=1,
    vmax=5,
    mask=peer_rank_matrix.isna(),
    linewidths=0.5,
    cbar_kws={
        "label": "Peer rank"
    }
)

plt.title(
    "Consistency of Illinois's Top-Five Economic Peers",
    fontsize=14
)

plt.xlabel("Research period")
plt.ylabel("Peer state")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 5H.10 — SAVE ILLINOIS PEER RESULTS
# ============================================================

illinois_peer_distances.to_csv(
    PROCESSED_DIR
    / "illinois_all_state_peer_distances.csv",
    index=False
)

illinois_top_peers.to_csv(
    TABLES_DIR
    / "illinois_top_five_peers_by_period.csv",
    index=False
)

illinois_peer_summary.to_csv(
    TABLES_DIR
    / "illinois_peer_summary_by_period.csv",
    index=False
)

recurring_illinois_peers.to_csv(
    TABLES_DIR
    / "illinois_recurring_top_peers.csv",
    index=False
)

illinois_peer_overlap.to_csv(
    TABLES_DIR
    / "illinois_peer_group_overlap.csv",
    index=False
)

print("Step 5H peer-analysis outputs saved successfully.")

In [ ]:
display(
    illinois_top_peers[
        [
            "Period_Label",
            "Peer_Rank",
            "state",
            "PCA_Distance",
            "Same_Cluster_as_Illinois",
            "Cluster_Name"
        ]
    ].round(4)
)

display(
    illinois_peer_summary[
        [
            "Period_Label",
            "Closest_Peer",
            "Closest_Distance",
            "Top_Five_Peers"
        ]
    ].round(4)
)

display(recurring_illinois_peers.round(4))
display(illinois_peer_overlap.round(4))

## Step 5H conclusions

**New Hampshire** was Illinois's closest economic peer `during all three
research periods`. Its PCA distance from Illinois declined from 1.5679
during the 2015–2019 baseline period to 1.0282 during 2020–2022 and
0.5193 during 2023–2024. This pattern indicates increasing multivariate
structural similarity.

New Jersey and Maryland also appeared among Illinois's five closest peers
during all three periods. Together with New Hampshire, they form the most
consistent core of Illinois's economic peer group.

The baseline and shock-period top-five peer lists contained the same five
states: Colorado, Maryland, New Hampshire, New Jersey, and Virginia.
Their Jaccard similarity was 1.0000, although their rankings changed.

The `post-shock` peer group changed more substantially. **New Hampshire,
New Jersey, and Maryland remained**, while Texas and Delaware replaced
Colorado and Virginia. The baseline-to-post-shock and shock-to-post-shock
Jaccard similarities were both 0.4286.

**Illinois** remained in Cluster **0—the High-Capacity Professional-Service
Economies cluster—during all three periods**. Nevertheless, the change in
its closest peers shows that stable broad cluster membership does not
necessarily imply an unchanged position within the cluster.

The PCA distances are relative multivariate measures without natural
economic units. A smaller distance indicates greater similarity across
PC1–PC4, but it does not mean that two states have identical economies
or establish a causal relationship.

| Analysis    | Comparison being made                         | Meaning                                                          |
| ----------- | --------------------------------------------- | ---------------------------------------------------------------- |
| Cluster map | Each state versus the seven cluster centroids | Which broad economic-structure segment best represents the state |
| Step 5H     | Each state directly versus Illinois           | Which individual states are closest to Illinois in PC1–PC4 space |


Mathematically:
- Cluster assignment: nearest cluster centroid
- Illinois peer ranking: nearest Illinois observation

Therefore, a state can be Illinois’s closest individual peer while belonging to a different cluster.


# Step 5I — Hierarchical Clustering Evaluation

This step applies hierarchical clustering to the same four-dimensional
PCA space used by the existing K-means model.

The analysis has two parts:

1. **Pooled evaluation:** Use all 150 state-period observations to compare
   hierarchical clustering with the existing seven-cluster K-means model.
2. **Period-specific evaluation:** Apply hierarchical clustering separately
   to the baseline, shock, and post-shock periods.

The pooled analysis provides the fairest direct comparison with K-means
because the original K-means model was fitted using all 150 observations.

The period-specific models are supplementary robustness checks. Their
cluster labels are not directly comparable across periods because each
period is fitted independently.

In [ ]:
# ============================================================
# STEP 5I.1 — IMPORT HIERARCHICAL CLUSTERING LIBRARIES
# ============================================================

from scipy.cluster.hierarchy import (
    linkage,
    dendrogram,
    cophenet
)

from scipy.spatial.distance import pdist

from sklearn.cluster import AgglomerativeClustering

from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score
)

print(
    "Hierarchical clustering libraries "
    "imported successfully."
)

In [ ]:
# ============================================================
# STEP 5I.2 — PREPARE HIERARCHICAL CLUSTERING DATA
# ============================================================

required_hierarchical_columns = (
    IDENTIFIER_COLUMNS
    + PCA_COMPONENT_NAMES
    + ["Cluster_ID"]
)

missing_hierarchical_columns = [
    column
    for column in required_hierarchical_columns
    if column not in cluster_assignments.columns
]

if missing_hierarchical_columns:
    raise KeyError(
        "Missing required hierarchical-clustering columns: "
        f"{missing_hierarchical_columns}"
    )

hierarchical_data = (
    cluster_assignments[
        required_hierarchical_columns
    ]
    .copy()
    .sort_values(
        ["state_fips", "period"]
    )
    .reset_index(drop=True)
)

# Four-dimensional PCA input used for clustering
X_hierarchical = (
    hierarchical_data[
        PCA_COMPONENT_NAMES
    ]
    .to_numpy(dtype=float)
)

# Existing K-means assignments
kmeans_labels = (
    hierarchical_data[
        "Cluster_ID"
    ]
    .to_numpy(dtype=int)
)

print(
    "Hierarchical input shape:",
    X_hierarchical.shape
)

print(
    "Existing K-means clusters:",
    np.unique(kmeans_labels)
)

display(
    hierarchical_data.head()
)

In [ ]:
# ============================================================
# STEP 5I.3 — VALIDATE MODEL INPUT
# ============================================================

assert hierarchical_data.shape[0] == 150

assert (
    hierarchical_data["state_fips"]
    .nunique()
    == 50
)

assert (
    hierarchical_data["period"]
    .nunique()
    == 3
)

assert (
    hierarchical_data["Cluster_ID"]
    .nunique()
    == 7
)

assert not hierarchical_data.duplicated(
    ["state_fips", "period"]
).any()

assert hierarchical_data[
    PCA_COMPONENT_NAMES
].notna().all().all()

assert np.isfinite(
    X_hierarchical
).all()

period_validation = (
    hierarchical_data
    .groupby(
        "period",
        observed=True
    )
    .agg(
        Number_of_States=(
            "state_fips",
            "nunique"
        ),
        Number_of_Observations=(
            "state",
            "size"
        ),
        KMeans_Clusters_Present=(
            "Cluster_ID",
            "nunique"
        )
    )
    .reindex(PERIOD_ORDER)
)

display(period_validation)

print(
    "All hierarchical-clustering "
    "input validations passed."
)

In [ ]:
# ============================================================
# STEP 5I.4 — CALCULATE EXISTING K-MEANS REFERENCE SCORES
# ============================================================

kmeans_reference = pd.DataFrame({
    "Model": [
        "K-means"
    ],
    "Linkage": [
        "Not applicable"
    ],
    "K": [
        hierarchical_data[
            "Cluster_ID"
        ].nunique()
    ],
    "Silhouette": [
        silhouette_score(
            X_hierarchical,
            kmeans_labels
        )
    ],
    "Calinski_Harabasz": [
        calinski_harabasz_score(
            X_hierarchical,
            kmeans_labels
        )
    ],
    "Davies_Bouldin": [
        davies_bouldin_score(
            X_hierarchical,
            kmeans_labels
        )
    ]
})

display(
    kmeans_reference.round(4)
)

In [ ]:
# ============================================================
# STEP 5I.5 — TEST POOLED HIERARCHICAL MODELS
# ============================================================

LINKAGE_METHODS = [
    "ward",
    "complete",
    "average",
    "single"
]

K_VALUES = range(2, 9)

hierarchical_model_records = []
hierarchical_label_lookup = {}

for linkage_method in LINKAGE_METHODS:

    for number_of_clusters in K_VALUES:

        hierarchical_model = (
            AgglomerativeClustering(
                n_clusters=number_of_clusters,
                linkage=linkage_method,
                metric="euclidean"
            )
        )

        hierarchical_labels = (
            hierarchical_model.fit_predict(
                X_hierarchical
            )
        )

        cluster_sizes = pd.Series(
            hierarchical_labels
        ).value_counts()

        model_key = (
            linkage_method,
            number_of_clusters
        )

        hierarchical_label_lookup[
            model_key
        ] = hierarchical_labels

        hierarchical_model_records.append({
            "Model":
                "Hierarchical",
            "Linkage":
                linkage_method.title(),
            "K":
                number_of_clusters,
            "Silhouette":
                silhouette_score(
                    X_hierarchical,
                    hierarchical_labels
                ),
            "Calinski_Harabasz":
                calinski_harabasz_score(
                    X_hierarchical,
                    hierarchical_labels
                ),
            "Davies_Bouldin":
                davies_bouldin_score(
                    X_hierarchical,
                    hierarchical_labels
                ),
            "Smallest_Cluster":
                int(cluster_sizes.min()),
            "Largest_Cluster":
                int(cluster_sizes.max())
        })

hierarchical_model_results = pd.DataFrame(
    hierarchical_model_records
)

display(
    hierarchical_model_results.round(4)
)

A clustering solution containing a one-observation cluster may represent an isolated state rather than a meaningful economic grouping. We will retain those results for review but exclude them from primary selection.

In [ ]:
# ============================================================
# STEP 5I.6 — RANK ELIGIBLE HIERARCHICAL MODELS
# ============================================================

hierarchical_model_results[
    "Eligible_for_Selection"
] = (
    hierarchical_model_results[
        "Smallest_Cluster"
    ].ge(2)
)

eligible_hierarchical_results = (
    hierarchical_model_results.loc[
        hierarchical_model_results[
            "Eligible_for_Selection"
        ]
    ]
    .copy()
)

eligible_hierarchical_results[
    "Silhouette_Rank"
] = (
    eligible_hierarchical_results[
        "Silhouette"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

eligible_hierarchical_results[
    "Calinski_Harabasz_Rank"
] = (
    eligible_hierarchical_results[
        "Calinski_Harabasz"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

eligible_hierarchical_results[
    "Davies_Bouldin_Rank"
] = (
    eligible_hierarchical_results[
        "Davies_Bouldin"
    ]
    .rank(
        ascending=True,
        method="min"
    )
)

eligible_hierarchical_results[
    "Mean_Metric_Rank"
] = (
    eligible_hierarchical_results[
        [
            "Silhouette_Rank",
            "Calinski_Harabasz_Rank",
            "Davies_Bouldin_Rank"
        ]
    ]
    .mean(axis=1)
)

hierarchical_ranked_results = (
    eligible_hierarchical_results
    .sort_values(
        [
            "Mean_Metric_Rank",
            "Silhouette"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

display(
    hierarchical_ranked_results.head(10).round(4)
)

In [ ]:
# ============================================================
# STEP 5I.7 — IDENTIFY THE BEST POOLED MODEL
# ============================================================

best_hierarchical_result = (
    hierarchical_ranked_results.iloc[0]
)

BEST_HIERARCHICAL_LINKAGE = (
    best_hierarchical_result[
        "Linkage"
    ].lower()
)

BEST_HIERARCHICAL_K = int(
    best_hierarchical_result[
        "K"
    ]
)

best_hierarchical_labels = (
    hierarchical_label_lookup[
        (
            BEST_HIERARCHICAL_LINKAGE,
            BEST_HIERARCHICAL_K
        )
    ]
)

print("Best pooled hierarchical model")
print("--------------------------------")
print(
    "Linkage:",
    BEST_HIERARCHICAL_LINKAGE.title()
)
print(
    "Number of clusters:",
    BEST_HIERARCHICAL_K
)
print(
    "Silhouette:",
    round(
        best_hierarchical_result[
            "Silhouette"
        ],
        4
    )
)
print(
    "Calinski–Harabasz:",
    round(
        best_hierarchical_result[
            "Calinski_Harabasz"
        ],
        4
    )
)
print(
    "Davies–Bouldin:",
    round(
        best_hierarchical_result[
            "Davies_Bouldin"
        ],
        4
    )
)
print(
    "Smallest cluster:",
    int(
        best_hierarchical_result[
            "Smallest_Cluster"
        ]
    )
)

The existing K-means solution uses seven clusters. Therefore, the fairest comparison is K-means (k=7) versus hierarchical clustering (k=7).

In [ ]:
# ============================================================
# STEP 5I.8 — SELECT THE BEST HIERARCHICAL K=7 MODEL
# ============================================================

hierarchical_k7_results = (
    hierarchical_model_results.loc[
        (
            hierarchical_model_results["K"].eq(7)
            & hierarchical_model_results[
                "Eligible_for_Selection"
            ]
        )
    ]
    .copy()
)

hierarchical_k7_results[
    "Silhouette_Rank"
] = (
    hierarchical_k7_results[
        "Silhouette"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

hierarchical_k7_results[
    "Calinski_Harabasz_Rank"
] = (
    hierarchical_k7_results[
        "Calinski_Harabasz"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

hierarchical_k7_results[
    "Davies_Bouldin_Rank"
] = (
    hierarchical_k7_results[
        "Davies_Bouldin"
    ]
    .rank(
        ascending=True,
        method="min"
    )
)

hierarchical_k7_results[
    "Mean_Metric_Rank"
] = (
    hierarchical_k7_results[
        [
            "Silhouette_Rank",
            "Calinski_Harabasz_Rank",
            "Davies_Bouldin_Rank"
        ]
    ]
    .mean(axis=1)
)

hierarchical_k7_results = (
    hierarchical_k7_results
    .sort_values(
        [
            "Mean_Metric_Rank",
            "Silhouette"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

display(
    hierarchical_k7_results.round(4)
)

In [ ]:
# ============================================================
# STEP 5I.9 — COMPARE K-MEANS WITH HIERARCHICAL K=7
# ============================================================

best_hierarchical_k7 = (
    hierarchical_k7_results.iloc[0]
)

best_k7_linkage = (
    best_hierarchical_k7[
        "Linkage"
    ].lower()
)

best_hierarchical_k7_labels = (
    hierarchical_label_lookup[
        (
            best_k7_linkage,
            7
        )
    ]
)

hierarchical_k7_comparison_row = pd.DataFrame({
    "Model": [
        "Hierarchical"
    ],
    "Linkage": [
        best_hierarchical_k7[
            "Linkage"
        ]
    ],
    "K": [
        7
    ],
    "Silhouette": [
        best_hierarchical_k7[
            "Silhouette"
        ]
    ],
    "Calinski_Harabasz": [
        best_hierarchical_k7[
            "Calinski_Harabasz"
        ]
    ],
    "Davies_Bouldin": [
        best_hierarchical_k7[
            "Davies_Bouldin"
        ]
    ]
})

kmeans_hierarchical_comparison = pd.concat(
    [
        kmeans_reference,
        hierarchical_k7_comparison_row
    ],
    ignore_index=True
)

display(
    kmeans_hierarchical_comparison.round(4)
)

kmeans_hierarchical_ari = (
    adjusted_rand_score(
        kmeans_labels,
        best_hierarchical_k7_labels
    )
)

print(
    "Adjusted Rand Index between "
    "K-means and hierarchical clustering:",
    round(
        kmeans_hierarchical_ari,
        4
    )
)

## Interpretations: 

- Higher sihouette: better-defined clusters.
- Higher Calinski-Harabasz: stronger separation.
- Lower Davies-Bouldin:lower overlap.
- Higher ARI: greater agreement between the two algorithms.
- ARI measures agreement, not which model is better. 

In [ ]:
# ============================================================
# STEP 5I.10 — TEST PERIOD-SPECIFIC HIERARCHICAL MODELS
# ============================================================

period_hierarchical_records = []
period_label_lookup = {}

for period in PERIOD_ORDER:

    period_data = (
        hierarchical_data.loc[
            hierarchical_data[
                "period"
            ].astype(str).eq(period)
        ]
        .copy()
        .reset_index(drop=True)
    )

    X_period = (
        period_data[
            PCA_COMPONENT_NAMES
        ]
        .to_numpy(dtype=float)
    )

    assert X_period.shape == (50, 4)

    for linkage_method in LINKAGE_METHODS:

        for number_of_clusters in K_VALUES:

            period_model = (
                AgglomerativeClustering(
                    n_clusters=number_of_clusters,
                    linkage=linkage_method,
                    metric="euclidean"
                )
            )

            period_labels = (
                period_model.fit_predict(
                    X_period
                )
            )

            cluster_sizes = pd.Series(
                period_labels
            ).value_counts()

            period_key = (
                period,
                linkage_method,
                number_of_clusters
            )

            period_label_lookup[
                period_key
            ] = period_labels

            period_hierarchical_records.append({
                "Period":
                    period,
                "Linkage":
                    linkage_method.title(),
                "K":
                    number_of_clusters,
                "Silhouette":
                    silhouette_score(
                        X_period,
                        period_labels
                    ),
                "Calinski_Harabasz":
                    calinski_harabasz_score(
                        X_period,
                        period_labels
                    ),
                "Davies_Bouldin":
                    davies_bouldin_score(
                        X_period,
                        period_labels
                    ),
                "Smallest_Cluster":
                    int(cluster_sizes.min()),
                "Largest_Cluster":
                    int(cluster_sizes.max()),
                "Eligible_for_Selection":
                    bool(
                        cluster_sizes.min() >= 2
                    )
            })

period_hierarchical_results = pd.DataFrame(
    period_hierarchical_records
)

display(
    period_hierarchical_results.head().round(4)
)

In [ ]:
# ============================================================
# STEP 5I.11 — RANK MODELS WITHIN EACH PERIOD
# ============================================================

eligible_period_results = (
    period_hierarchical_results.loc[
        period_hierarchical_results[
            "Eligible_for_Selection"
        ]
    ]
    .copy()
)

eligible_period_results[
    "Silhouette_Rank"
] = (
    eligible_period_results
    .groupby("Period")[
        "Silhouette"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

eligible_period_results[
    "Calinski_Harabasz_Rank"
] = (
    eligible_period_results
    .groupby("Period")[
        "Calinski_Harabasz"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

eligible_period_results[
    "Davies_Bouldin_Rank"
] = (
    eligible_period_results
    .groupby("Period")[
        "Davies_Bouldin"
    ]
    .rank(
        ascending=True,
        method="min"
    )
)

eligible_period_results[
    "Mean_Metric_Rank"
] = (
    eligible_period_results[
        [
            "Silhouette_Rank",
            "Calinski_Harabasz_Rank",
            "Davies_Bouldin_Rank"
        ]
    ]
    .mean(axis=1)
)

period_hierarchical_ranked = (
    eligible_period_results
    .sort_values(
        [
            "Period",
            "Mean_Metric_Rank",
            "Silhouette"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

best_period_hierarchical_models = (
    period_hierarchical_ranked
    .groupby(
        "Period",
        observed=True,
        as_index=False
    )
    .first()
    .set_index("Period")
    .reindex(PERIOD_ORDER)
    .reset_index()
)

display(
    best_period_hierarchical_models[
        [
            "Period",
            "Linkage",
            "K",
            "Silhouette",
            "Calinski_Harabasz",
            "Davies_Bouldin",
            "Smallest_Cluster",
            "Largest_Cluster",
            "Mean_Metric_Rank"
        ]
    ].round(4)
)

### 1. Direct comparison at k=7

| Metric             |     K-means | Hierarchical Ward | Better model |
| ------------------ | ----------: | ----------------: | ------------ |
| Silhouette         |  **0.3076** |            0.2841 | K-means      |
| Calinski–Harabasz  | **73.6397** |           67.0326 | K-means      |
| Davies–Bouldin     |  **1.0626** |            1.0840 | K-means      |
| Number of clusters |           7 |                 7 | Same         |


### 2. Agreement between the models

The Adjusted Rand Index is:

***ARI*** = 0.7458

This indicates relatively high agreement between K-means and hierarchical clustering. The algorithms do not produce identical assignments, but they detect broadly similar economic structures.

This supports the robustness of the seven-cluster solution.

### 3. Best pooled hierarchical model 

The ranking selected:
- Ward linkage;
- k=3 
- silhouette = 0.3576;
- Calinski–Harabasz = 66.1940;
- Davies–Bouldin = 0.8972.

However, the cluster sizes are:
- smallest cluster: 9 observations;
- largest cluster: 107 observations.

Approximately 71% of all observations are in one cluster. Therefore, although Ward **k=3** has better silhouette and Davies–Bouldin results, it produces a much broader and less detailed grouping than K-means **k=7**.

It should not automatically be considered the superior model because it answers a different question:
- **k=3**: broad economic groupings;
- **k=7**: more detailed economic structures.

### 4. Period-specific models

| Period     | Best linkage | \(k\) | Silhouette | Cluster sizes | Interpretation                 |
| ---------- | ------------ | ----: | ---------: | ------------: | ------------------------------ |
| Baseline   | Average      |     3 |     0.3522 |          3–37 | One dominant cluster           |
| Shock      | Average      |     2 |     0.3628 |         12–38 | Primarily a two-group division |
| Post-shock | Ward         |     4 |     0.3234 |          3–27 | More differentiated structure  |

These results suggest that the hierarchical structure may have become more differentiated after the shock:
- Baseline: three broad groups;
- Shock: two broad groups;
- Post-shock: four groups.

However, we should describe this as an exploratory finding. Because the three models were fitted independently and use different values of **k**, their cluster numbers cannot be interpreted as matching categories across periods.

### Conclusion: 

The ***seven-cluster K-means model*** outperformed the comparable ***seven-cluster hierarchical model*** across all three internal evaluation metrics. However, the Adjusted Rand Index of 0.7458 indicates substantial agreement between the two algorithms. Hierarchical clustering therefore supports the broad structural patterns identified by K-means, although it favors fewer and more unevenly sized clusters. **K-means remains the primary model**, while hierarchical clustering serves as a robustness and visualization method


In [ ]:
# ============================================================
# STEP 5I.12 — COMPARE K-MEANS AND HIERARCHICAL MEMBERSHIP
# ============================================================

hierarchical_data[
    "Hierarchical_Ward_K7"
] = best_hierarchical_k7_labels

kmeans_hierarchical_crosstab = pd.crosstab(
    hierarchical_data["Cluster_ID"],
    hierarchical_data["Hierarchical_Ward_K7"],
    rownames=["K-means Cluster"],
    colnames=["Hierarchical Ward Cluster"]
)

display(kmeans_hierarchical_crosstab)

In [ ]:
# ============================================================
# STEP 5I.13 — POOLED WARD DENDROGRAM
# ============================================================

def calculate_cut_threshold(
    linkage_matrix,
    number_of_clusters
):
    lower_distance = linkage_matrix[
        -number_of_clusters,
        2
    ]

    upper_distance = linkage_matrix[
        -(number_of_clusters - 1),
        2
    ]

    return (
        lower_distance
        + upper_distance
    ) / 2


pooled_ward_linkage = linkage(
    X_hierarchical,
    method="ward",
    metric="euclidean",
    optimal_ordering=True
)

pooled_cophenetic_correlation, _ = cophenet(
    pooled_ward_linkage,
    pdist(
        X_hierarchical,
        metric="euclidean"
    )
)

pooled_k7_cut = calculate_cut_threshold(
    pooled_ward_linkage,
    number_of_clusters=7
)

plt.figure(figsize=(16, 8))

dendrogram(
    pooled_ward_linkage,
    truncate_mode="lastp",
    p=30,
    show_leaf_counts=True,
    leaf_rotation=90,
    leaf_font_size=9,
    color_threshold=pooled_k7_cut
)

plt.axhline(
    y=pooled_k7_cut,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Seven-cluster cut"
)

plt.title(
    "Pooled Hierarchical Structure: "
    "Ward Linkage Across 150 State-Period Observations",
    fontsize=15
)

plt.xlabel(
    "State-period observations or merged branches"
)

plt.ylabel("Ward linkage distance")

plt.legend()
plt.tight_layout()
plt.show()

print(
    "Pooled cophenetic correlation:",
    round(
        pooled_cophenetic_correlation,
        4
    )
)

This uses the selected configuration for each period:
- Baseline: Average, \(k=3\)
- Shock: Average, \(k=2\)
- Post-shock: Ward, k=4

In [ ]:
# ============================================================
# STEP 5I.14 — CREATE PERIOD-SPECIFIC DENDROGRAMS
# ============================================================

period_dendrogram_summary = []

fig, axes = plt.subplots(
    3,
    1,
    figsize=(18, 24)
)

for axis, period in zip(
    axes,
    PERIOD_ORDER
):

    selected_result = (
        best_period_hierarchical_models.loc[
            best_period_hierarchical_models[
                "Period"
            ].astype(str).eq(period)
        ]
        .iloc[0]
    )

    selected_linkage = (
        selected_result["Linkage"]
        .lower()
    )

    selected_k = int(
        selected_result["K"]
    )

    period_data = (
        hierarchical_data.loc[
            hierarchical_data[
                "period"
            ].astype(str).eq(period)
        ]
        .copy()
        .sort_values("state")
        .reset_index(drop=True)
    )

    X_period = (
        period_data[
            PCA_COMPONENT_NAMES
        ]
        .to_numpy(dtype=float)
    )

    period_linkage_matrix = linkage(
        X_period,
        method=selected_linkage,
        metric="euclidean",
        optimal_ordering=True
    )

    period_distances = pdist(
        X_period,
        metric="euclidean"
    )

    cophenetic_correlation, _ = cophenet(
        period_linkage_matrix,
        period_distances
    )

    cut_threshold = calculate_cut_threshold(
        period_linkage_matrix,
        selected_k
    )

    dendrogram(
        period_linkage_matrix,
        labels=period_data[
            "state"
        ].tolist(),
        leaf_rotation=90,
        leaf_font_size=8,
        color_threshold=cut_threshold,
        ax=axis
    )

    axis.axhline(
        y=cut_threshold,
        color="black",
        linestyle="--",
        linewidth=1.3,
        label=f"{selected_k}-cluster cut"
    )

    axis.set_title(
        (
            f"{PERIOD_DISPLAY_LABELS[period]} — "
            f"{selected_linkage.title()} linkage, "
            f"k={selected_k}"
        ),
        fontsize=14
    )

    axis.set_xlabel("State")
    axis.set_ylabel("Linkage distance")
    axis.legend(loc="upper right")

    period_dendrogram_summary.append({
        "Period":
            period,
        "Linkage":
            selected_linkage.title(),
        "K":
            selected_k,
        "Cophenetic_Correlation":
            cophenetic_correlation,
        "Cut_Threshold":
            cut_threshold
    })

plt.suptitle(
    "Hierarchical Economic Structures by Period",
    fontsize=17,
    y=1.01
)

plt.tight_layout()
plt.show()

period_dendrogram_summary = pd.DataFrame(
    period_dendrogram_summary
)

display(
    period_dendrogram_summary.round(4)
)